## Inverse Sampling the DF of a DREAMS galaxy with Eddington formula

The distribution function of a galaxy describes the probability of finding a star with given phase-space coordinates $(\bm{x},\bm{y})$. For a stellar system in a spherical potential, the distribution function can be derived directly from the hamiltonian of the system using **Eddington's formula** (Eq. 4.46a in Binney & Tremaine's Galactic Dynamics):

$$
f(\mathcal{E}) = \frac{1}{\sqrt{8}\pi^2} \left[ \int_0^{\mathcal{E}} \frac{d^2\rho}{d\Psi^2} \frac{d\Psi}{\sqrt{\mathcal{E} - \Psi}} + \frac{1}{\sqrt{\mathcal{E}}} \left( \frac{d\rho}{d\Psi} \right)_{\Psi=0} \right],
$$

where $\epsilon = - \Psi - \frac{1}{2}v^{2} + \Psi_{0}$ is the relative energy of the particles and $\Psi_{0}$ is a constant term which ensures that $f>0$ for $\epsilon>0$ and $f=0$ when $\epsilon \leq 0$.

For most physically realistic galaxies where density smoothly approaches zero at infinity (where $\Psi \to 0$), the boundary term vanishes, simplifying the equation to:

$$
f(\mathcal{E}) = \frac{1}{\sqrt{8}\pi^2} \int_0^{\mathcal{E}} \frac{d^2\rho}{d\Psi^2} \frac{d\Psi}{\sqrt{\mathcal{E} - \Psi}}
$$


In our scenario, the density and potential fields of the galaxy are approximated across a radial grid using basis-function expansion with $\mathtt{EXP}$. Therefore, $\rho$, $\Psi$, and $\epsilon$ are all functions of the positions ($\bm{x}$) and velocities ($\bm{v}$) of the particles in the simulation. 



In [45]:
import astropy.units as u
from gala.dynamics import mockstream as ms
import gala.potential as gp
import gala.dynamics as gd
import gala.integrate as gi
import matplotlib.pyplot as plt
import numpy as np
from scipy.differentiate import derivative
import sys
import time

sys.path.append("/mnt/home/asante/streams_in_dreams/")
import EXP4DREAMS

snap_path = "/mnt/home/dreams/ceph/Sims/CDM/MW_zooms/SB5/box_695/"
group_path = "/mnt/home/dreams/ceph/FOF_Subfind/CDM/MW_zooms/SB5/box_695/" 
output_path = "/mnt/home/asante/ceph/streams_test/"
z_range = [1,0]

basis_dict = {# PartType : basis params 
                1: {"Lmax": 6, "nmax": 20, "numr": 2000, "rmin": 0.001, "rmax":1.5},
                4: {"mmax": 6, "nmax": 12 }
                }
density_dict = {1: {"bins": 500, 
                    "rangevals": [0.1, 100] 
                    },
                4: {}
                }

# Load the Simulation data
MW_sim = EXP4DREAMS.DREAMSMW(snap_path=snap_path,
                             group_path=group_path)

# Perform EXP expansion
EXP_gen = EXP4DREAMS.EXPBFE_builder(sim=MW_sim,
                                    basis_params_dict=basis_dict,
                                    density_dict=density_dict,
                                    z_range=z_range,
                                    output_dir=output_path)

# Create Gala-EXP potential object
pot, exp_units = EXP_gen.build_gala_potential()


               Scale Radius (DM): 		4.5 kpc
               Virial Radius: 		152.4 kpc
               Virial Mass: 		1e+12 M_sun
               Disc Scale Radius: 		2.1 kpc
               Disc Scale Height: 		0.5 kpc
               
Indexing snapshot directory...
Found 5 snapshots in range [1, 0] (Snap 61 to 90)
Snapshots used for the expansion: 61 to 90
Building basis for the expansion...
! Scaling:  R= 0.6514448938042994   M= 1.0
1.0 1.4436769529029623 1.4436769529029623 1.4436769529029623
{'id': 'sphereSL', 'parameters': {'numr': 2000, 'rmin': 0.001, 'rmax': 1.5, 'Lmax': 6, 'nmax': 20, 'rmapping': 0.02931458978538835, 'modelname': '/mnt/home/asante/ceph/streams_test/basis_empirical_PartType1.txt', 'cachename': '/mnt/home/asante/ceph/streams_test/basis_empirical_PartType1.cache.run0', 'pcavar': True, 'subsamp': 1000}, 'runtag': 'run0'}


---- SLGridSph::ReadH5Cache: error reading </mnt/home/asante/ceph/streams_test/basis_empirical_PartType1.cache.run0>
---- SLGridSph::ReadH5Cache: HDF5 error is <Unable to open file /mnt/home/asante/ceph/streams_test/basis_empirical_PartType1.cache.run0 (File accessibility) Unable to open file>


---- SLGridSph::WriteH5Cache: wrote </mnt/home/asante/ceph/streams_test/basis_empirical_PartType1.cache.run0>
---- Spherical::orthoTest: worst=0.000977564
{'id': 'cylinder', 'parameters': {'acyl': 0.013667086216408546, 'hcyl': 0.002966608820026228, 'lmaxfid': 32, 'nmaxfid': 32, 'mmax': 6, 'nmax': 12, 'ncylnx': 256, 'ncylny': 128, 'ncylodd': 3, 'rnum': 1000, 'pnum': 0, 'tnum': 80, 'ashift': 0.5, 'vflag': 16, 'logr': False, 'cachename': '/mnt/home/asante/ceph/streams_test/basis_empirical_PartType4.cache.run0', 'pcavar': True, 'subsamp': 1000}}
Calculating the coefficients at snapshots: [61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90]
---- EmpCylSL cache parameter ascl: wanted 0.0136671 found 0.0136671
---- DiskType is <exponential>
---- pyEXP uses sech^2(z/h) rather than the more common sech^2(z/(2h))
---- Use the 'sech2: true' in your YAML config to use sech^2(z/(2h))
---- pyEXP will assume sech^2(z/(2h)) by default

Worker    0: tables allocated, MMAX=6
---- EmpCylSL::cache_grid: HDF5 parameter mismatch


Snapshot 62 not found
Snapshot 63 not found
Snapshot 64 not found
Snapshot 65 not found
Snapshot 66 not found
Snapshot 68 not found
Snapshot 69 not found
Snapshot 70 not found
Snapshot 71 not found
Snapshot 72 not found
Snapshot 74 not found
Snapshot 75 not found
Snapshot 76 not found
Snapshot 77 not found
Snapshot 78 not found
Snapshot 79 not found
Snapshot 80 not found
Snapshot 82 not found
Snapshot 83 not found
Snapshot 84 not found
Snapshot 85 not found
Snapshot 86 not found
Snapshot 87 not found
Snapshot 88 not found
Snapshot 89 not found
Time taken for PartType 1: 1.19 minutes
Snapshot 62 not found
Snapshot 63 not found
Snapshot 64 not found
Snapshot 65 not found
Snapshot 66 not found
Snapshot 68 not found
Snapshot 69 not found
Snapshot 70 not found
Snapshot 71 not found
Snapshot 72 not found
Snapshot 74 not found
Snapshot 75 not found
Snapshot 76 not found
Snapshot 77 not found
Snapshot 78 not found
Snapshot 79 not found
Snapshot 80 not found
Snapshot 82 not found
Snapshot 83 no

Worker    0: tables allocated, MMAX=6


In [151]:
# Density function from EXP
def rho(r):
    
    basis = EXP_gen.basis[1]
    coefs = EXP_gen.coefs[1]
    
    # Load basis coefficients
    basis.set_coefs(coefs.getCoefStruct(EXP_gen.coefs[1].Times()[-1]))
    idx = basis.getFieldLabels().index("dens")
    
    r_flat = np.asarray(r).flatten()
    vals = []
    for xi in r_flat:
        vals.append(basis.getFields(xi, 0.0, 0.0)[idx])
        
    return np.array(vals).reshape(np.shape(r))
    
# Potential function EXP
def psi(r):
    
    basis = EXP_gen.basis[1]
    coefs = EXP_gen.coefs[1]
    
    # Load basis coefficients
    basis.set_coefs(coefs.getCoefStruct(EXP_gen.coefs[1].Times()[-1]))
    idx = basis.getFieldLabels().index("potl")
    
    r_flat = np.asarray(r).flatten()
    vals = []
    for xi in r_flat:
        vals.append(basis.getFields(xi, 0.0, 0.0)[idx])
        
    return np.array(vals).reshape(np.shape(r))
 
    
def epsilon(r, v, phi0=0.):

    # Compute the relative energy at radius r
    H = psi(r) + 0.5*v*v
    
    return - H - phi0



To evaluate the integral computationally on this spatial grid, the variable of integration must be transformed from $d\Psi$ to $dr$. The first derivative of density with respect to the potential is expanded via the chain rule:
$$
\frac{d\rho}{d\Psi} = \frac{d\rho/dr}{d\Psi/dr} = \frac{\rho'(r)}{\Psi'(r)}
$$.

The differential in the potential is transformed in terms of the radial position as:

$$
\frac{d}{d\Psi} = \left ( \frac{d\Psi}{dr} \right)^{-1} \, \frac{d}{dr} = \Psi'(r)^{-1} \frac{d}{dr}.
$$

Therefore, the second derivative term becomes:

$$

\frac{d^2\rho}{d\Psi^2} = \frac{\rho''(r) \, \Psi'(r) - \rho'(r) \, \Psi''(r)}{\Psi'(r)^3}

$$

In [178]:
def drho_dr(r):
    
    basis = EXP_gen.basis[1]
    coefs = EXP_gen.coefs[1]
    
    # Load basis coefficients
    basis.set_coefs(coefs.getCoefStruct(EXP_gen.coefs[1].Times()[-1]))
    idx = basis.getFieldLabels().index("dens")
    
    
    def dens_fn(x):
        # 1. Flatten SciPy's multi-dimensional step arrays into a safe 1D list
        x_flat = np.asarray(x).flatten()
        vals = []
        
        for xi in x_flat:
            
            vals.append(basis.getFields(xi, 0.0, 0.0)[idx])
                
        # 4. Reshape the flat results back to the exact shape SciPy needs
        return np.array(vals).reshape(np.shape(x))
    
    
    derivative_results = derivative(dens_fn, 
                                    r,
                                    maxiter=100,
                                    initial_step=(0.1*u.kpc).to(exp_units['length']).value
                                    )

    
    return derivative_results.df

def dpsi_dr(r):
    # This is just the radial force, which can be directly obtained from EXP
    basis = EXP_gen.basis[1]
    coefs = EXP_gen.coefs[1]
    
    # Load basis coefficients
    basis.set_coefs(coefs.getCoefStruct(EXP_gen.coefs[1].Times()[-1]))
    idx = basis.getFieldLabels().index("rad force")
    
    r_flat = np.asarray(r).flatten()
    vals = []
    for xi in r_flat:
        vals.append(basis.getFields(xi, 0.0, 0.0)[idx])
        
    return np.array(vals).reshape(np.shape(r))
    
# The second derivative of the density and potential fields are computed numerically
def drho2_dr2(r):
    
    basis = EXP_gen.basis[1]
    coefs = EXP_gen.coefs[1]
    
    # Load basis coefficients
    basis.set_coefs(coefs.getCoefStruct(EXP_gen.coefs[1].Times()[-1]))
    idx = basis.getFieldLabels().index('dens')
    
    def get_field_derivative(r):
    
        def dens_fn(x):
            # 1. Flatten SciPy's multi-dimensional step arrays into a safe 1D list
            x_flat = np.asarray(x).flatten()
            vals = []
            
            for xi in x_flat:
                
                vals.append(basis.getFields(xi, 0.0, 0.0)[idx])
                    
            # 4. Reshape the flat results back to the exact shape SciPy needs
            return np.array(vals).reshape(np.shape(x))
    
    
        derivative_results = derivative(dens_fn, 
                                        r,
                                        maxiter=100,
                                        initial_step=(0.1*u.kpc).to(exp_units['length']).value
                                        )
        
        return derivative_results.df
    
    
    derivative_results = derivative(get_field_derivative,
                                    r,
                                    maxiter=100,
                                    initial_step=(0.1*u.kpc).to(exp_units['length']).value
                                    )
    
    return derivative_results.df

def dpsi2_dr2(r):
        
    basis = EXP_gen.basis[1]
    coefs = EXP_gen.coefs[1]
    
    # Load basis coefficients
    basis.set_coefs(coefs.getCoefStruct(EXP_gen.coefs[1].Times()[-1]))
    idx = basis.getFieldLabels().index("rad force")
    
    
    def dens_fn(x):
        # 1. Flatten SciPy's multi-dimensional step arrays into a safe 1D list
        x_flat = np.asarray(x).flatten()
        vals = []
        
        for xi in x_flat:
            
            vals.append(basis.getFields(xi, 0.0, 0.0)[idx])
                
        # 4. Reshape the flat results back to the exact shape SciPy needs
        return np.array(vals).reshape(np.shape(x))
    
    
    derivative_results = derivative(dens_fn, 
                                    r,
                                    maxiter=100,
                                    initial_step=(0.1*u.kpc).to(exp_units['length']).value
                                    )

    
    return derivative_results.df

def drho2_dpsi2(r):
    
    numerator = drho2_dr2(r) * dpsi_dr(r) - drho_dr(r) * dpsi2_dr2(r) 
    denominator = dpsi_dr(r)**3 + 1e-12
    
    return numerator / denominator

In [153]:
r = np.array([(40*u.kpc).to(exp_units['length']).value,
                                 (40*u.kpc).to(exp_units['length']).value])

v = np.array([(200*u.km/u.s).to(exp_units['velocity']).value,
                (200*u.km/u.s).to(exp_units['velocity']).value])

Applying the change of variables and mapping the integration limits—where $\Psi=0$ corresponds to $r=\infty$, and $\Psi=\mathcal{E}$ corresponds to the turning point $r=r_{\mathcal{E}}$, Eddington's formula becomes: 

$$
f(\mathcal{r,v}) = -\frac{1}{\sqrt{8}\pi^2} \int_{r_{\mathcal{E}}}^{\infty} \frac{\rho''(r) \, \Psi'(r) - \rho'(r) \, \Psi''(r)}{\Psi'(r)^3} \frac{dr}{\sqrt{\mathcal{E}(r, v) - \Psi(r)}}.
$$

The turning point $r_{\mathcal{E}}$ is the maximum radial distance a particle bound to that system could have given its total energy. This can be calculated numerically using a root-finder for a function equating the $EXP$ potential to the relative energy of the particle

In [175]:
from scipy.optimize import root_scalar

def turning_radius(x,v):
    
    # Compute the turning radius for a given position and velocity
    r = np.linalg.norm(x, axis=-1)
    v_mag = np.linalg.norm(v, axis=-1)
    
    vals = []
    for xi,vi in zip(r,v_mag):
        try:
            r_eps = root_scalar(lambda x: epsilon(xi, vi) + psi(x), 
                                bracket=[1e-5, 10], method='brentq').root
        except ValueError as e:
            print(f"Error finding root for r={xi}, v={vi}: {e}")
        vals.append(r_eps)
        
    return np.array(vals)

In [187]:
from scipy.integrate import quad

def df(x,v):
    
    # Compute the distribution function using Eddington's formula
    r = np.linalg.norm(x, axis=-1)
    v_mag = np.linalg.norm(v, axis=-1)
    
    r_eps = turning_radius(x,v)
    
    def integrand(r,v_mag_scalar):
        
        delta = epsilon(r, v_mag_scalar) - psi(r)
        
        return np.where(delta > 0, drho2_dpsi2(r) / np.sqrt(delta), 0.0)
    
    vals = []
    for r0, vm in zip(r_eps, v_mag):

        f_eps = quad(integrand, float(r0), 2, 
                     args=(float(vm),), 
                     limit=100, epsabs=1e-5, epsrel=1e-5)[0]

        vals.append(f_eps / (np.sqrt(8)*np.pi**2))
    
    return np.array(vals)


In [188]:
# Create a grid of positions and velocities
r_grid = np.linspace(0.1, 100, 10) 
v_grid = np.linspace(0.1, 200, 10) 

# Compute the position and velocity vectors
x_grid = (np.array([[ri, 0, 0] for ri in r_grid])*u.kpc).to(exp_units['length']).value
v_grid = (np.array([[vi, 0, 0] for vi in v_grid])*u.km/u.s).to(exp_units['velocity']).value

In [189]:
np.linalg.norm(x_grid, axis=-1).shape

(10,)

In [ ]:
df(x_grid, v_grid)

/tmp/ipykernel_908129/3829425594.py:20: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  f_eps = quad(integrand, float(r0), 2,
/tmp/ipykernel_908129/3829425594.py:20: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  f_eps = quad(integrand, float(r0), 2,
/tmp/ipykernel_908129/3829425594.py:20: IntegrationWarning: The maximum number of subdivisions (100) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.